# QTagger+ — Final Training Notebook (Kaggle Edition)

**Original CIC-MalMem-2022 CSV → corrected QSVM v3.1 → evaluation → saved model → verified inference.**

This notebook is built from your own QIntern26 research, not a replacement architecture:

- [Week 1 MalMem training notebook](https://github.com/shanmukhdatta/Qintern26/blob/main/Week_1/code/MalMem_Training.ipynb): data inspection, classical baselines and metrics.
- [Corrected Week 6 QSVM v3.1 pipeline](https://github.com/shanmukhdatta/Qintern26/blob/main/Week_6/Day_5a/day5a_final/pipeline_v3_1_fixed.py): train-only filtering, signed-log, StandardScaler, XGBoost top-8, RobustScaler, corrected trainable local kernel, KTA, Nyström, SVM.
- [Earlier model reload example](https://github.com/shanmukhdatta/Qintern26/blob/main/Week_5/Day34_V3_Supplementary/model/load_and_predict_example.py): serialization concept only; it is tied to the older buggy model and is **not** reused directly.

**Read the Markdown immediately before each code cell.** Every action is separated, so you can learn the pipeline without first reading the code.

**Scope:** Default is the corrected v3.1 **three-malware-family** task (Ransomware, Spyware, Trojan). Optionally change `TASK="four_class"` to include Benign; this is a new experiment and does not reproduce historical accuracy. We train on **real** MalMem data, not CTGAN.

**Evaluation integrity:** Split before fitting preprocessors, remove exact feature duplicates, use matched quantum/classical rows, never tune using test labels. Local overlap readout may be asymmetric or indefinite; this is an explicit Nyström feature map with diagnostics, **not a proven PSD quantum kernel**.

**Execution status:** This notebook has been reviewed for Kaggle paths, step order, saved-model contents, and preflight assertions. Full Kaggle training and actual accuracy must still be verified by running it.

## Before running on Kaggle

1. Open Kaggle → **Code → New Notebook**, then use **File → Import Notebook** or upload this `.ipynb`.
2. In the right sidebar choose **Add Input**, attach a dataset containing the original file named **`MalMem2022.csv`**. If it is not attached, enable **Internet** and set `ALLOW_GITHUB_DOWNLOAD=True` in configuration.
3. Under Notebook Settings, enable **Internet** for the first-time PennyLane installation. A **CPU** runtime is suitable; a GPU is not required by this simulator.
4. Run the installation cell. If package versions change, restart the session and then **Run All**.
5. Inspect each printed checkpoint. Model and figures are written to **`/kaggle/working/qtagger_outputs/`**; the final cell makes a downloadable ZIP.

**Expected runtime:** The quantum step performs many circuit evaluations and can take substantial time. Start with `TRAIN_SUBSET=100`, `TEST_SUBSET=30`, `N_LANDMARKS=16` for a quick smoke test; restore the defaults for your main training. A smoke-test result is not your final accuracy.

**Reproducibility:** Use a fixed seed and record the package versions. A notebook that passes structural checks is not guaranteed to execute successfully in every Kaggle environment until actually run.

### 00 — Install quantum dependencies

**Purpose:** Install PennyLane and its optional faster CPU simulator. Kaggle normally already includes NumPy, pandas, scikit-learn, XGBoost and plotting libraries. **Expected output:** a successful installation message. **If this fails:** enable Internet in Kaggle Settings; restart the session after installation if prompted.

In [ ]:
%pip install -q "pennylane>=0.40,<0.45" "pennylane-lightning>=0.40,<0.45"

### 01 — Import libraries

**Purpose:** Import data processing, ML, quantum, visualization and model-saving packages. **Expected output:** no errors; if an import fails, install the missing package and restart the runtime.

In [ ]:
import json, os, sys, time, hashlib, platform, warnings
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import scipy
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
import xgboost
import sklearn
import pennylane as qml

### 02 — Configure training

**Purpose:** Choose the task, sample sizes, quantum architecture and random seed. **Expected output:** the chosen settings and package versions. **Tip:** keep `TASK='malware3'` to match the corrected original-data v3.1 loader; choose `four_class` only for a new experiment.

In [ ]:
SEED = 7
TASK = "malware3"      # "malware3" or "four_class"
N_QUBITS = 8
LAYERS = 2
TRAIN_SUBSET = 300     # use 100 for a quick smoke test
TEST_SUBSET = 120      # use 30 for a quick smoke test
ALIGN_SUBSET = 10
ALIGN_MAXITER = 12
N_LANDMARKS = 40       # use 16 for a quick smoke test
SVM_C = 1.0
TEST_FRACTION = 0.30
DROP_EXACT_DUPLICATES = True
ALLOW_GITHUB_DOWNLOAD = False  # set True only if Kaggle Internet is enabled
IS_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = (Path("/kaggle/working") if IS_KAGGLE else Path.cwd()) / "qtagger_outputs" / TASK
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Environment:", "Kaggle" if IS_KAGGLE else "Local Jupyter")
print("Task:", TASK, "| seed:", SEED, "| qubits:", N_QUBITS)
print("Quantum budget:", TRAIN_SUBSET, "train /", TEST_SUBSET, "test /", N_LANDMARKS, "landmarks")
print("Versions:", {"python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "xgboost": xgboost.__version__, "pennylane": qml.__version__})

### 03 — Find the original dataset

**Purpose:** Automatically locate `MalMem2022.csv` in Kaggle **Add Input**, your local repository, or a local working folder. **Expected output:** the full path. If missing, the error explains how to attach the file; optional GitHub download requires Internet.

In [ ]:
candidate_paths = []
for folder in [Path("/kaggle/input"), Path.cwd() / "Datasets_", Path.cwd(), Path("/kaggle/working")]:
    if folder.exists():
        candidate_paths.extend(folder.rglob("MalMem2022.csv"))
candidate_paths = list(dict.fromkeys(candidate_paths))
if candidate_paths:
    DATA_PATH = candidate_paths[0]
else:
    DATA_PATH = (Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "Datasets_") / "MalMem2022.csv"
    if ALLOW_GITHUB_DOWNLOAD:
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        url = "https://raw.githubusercontent.com/shanmukhdatta/Qintern26/main/Datasets_/MalMem2022.csv"
        print("Downloading the original MalMem CSV from your GitHub repository...")
        urlretrieve(url, DATA_PATH)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Original MalMem2022.csv not found. In Kaggle choose Add Input and attach the CSV, "
        "or enable Internet and set ALLOW_GITHUB_DOWNLOAD=True in cell 02.")
print("Found original dataset:", DATA_PATH)

### 04 — Read and fingerprint the CSV

**Purpose:** Load the real memory-feature dataset, confirm the expected `Category` column, and record its SHA-256 fingerprint. **Expected output:** shape, fingerprint, sample rows and label counts. This establishes which exact dataset version was used.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
assert "Category" in df.columns, "Expected the original MalMem2022.csv with Category column."
sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("CSV:", DATA_PATH, "| shape:", df.shape, "| SHA256:", sha)
print("Category examples:", df["Category"].value_counts().head(12).to_dict())
display(df.head(3))

### 05 — Create the prediction labels

**Purpose:** Convert the fine-grained `Category` names into malware-family labels. **Expected output:** selected classes and their counts. Three-family mode excludes Benign; four-class mode retains it.

In [ ]:
category = df["Category"].astype(str).str.strip()
family = category.str.split("-", n=1).str[0]
if TASK == "malware3":
    valid = family.isin(["Ransomware", "Spyware", "Trojan"])
    class_order = ["Ransomware", "Spyware", "Trojan"]
elif TASK == "four_class":
    valid = family.isin(["Benign", "Ransomware", "Spyware", "Trojan"])
    class_order = ["Benign", "Ransomware", "Spyware", "Trojan"]
else:
    raise ValueError("TASK must be malware3 or four_class")


print("Requested classes:", class_order)
print("Selected rows:", int(valid.sum()), "| counts:", family.loc[valid].value_counts().to_dict())

### 06 — Select numerical features

**Purpose:** Keep numerical memory-forensic features and remove label/identifier columns such as `Category`, `Class` and `Filename`. **Expected output:** the number of raw feature columns and missing-value counts. These same columns will be required for inference.

In [ ]:
working = df.loc[valid].copy().reset_index(drop=True)
labels = family.loc[valid].reset_index(drop=True)
excluded = {"Class", "Category", "Filename", "Family", "fam"}
raw_feature_cols = [c for c in working.select_dtypes(include=[np.number]).columns if c not in excluded]
assert len(raw_feature_cols) >= N_QUBITS, "Too few numeric features."
X_raw = working[raw_feature_cols].replace([np.inf, -np.inf], np.nan)
assert not X_raw.isna().all(axis=0).any(), "A numeric feature is entirely missing."

print("Raw numerical features:", len(raw_feature_cols))
print("Rows with missing numeric features:", int(X_raw.isna().any(axis=1).sum()))

### 07 — Check and remove duplicate feature records

**Purpose:** Prevent identical numerical feature vectors from appearing in both training and test data. If an identical vector has conflicting family labels, remove all those ambiguous records rather than arbitrarily choosing one label. **Expected output:** duplicate and conflict counts, followed by class counts after cleanup.

In [ ]:
feature_hash = pd.util.hash_pandas_object(X_raw, index=False)
label_count_by_hash = pd.DataFrame({"hash": feature_hash, "label": labels}).groupby("hash")["label"].nunique()
conflict_hashes = label_count_by_hash[label_count_by_hash > 1].index
conflict_mask = feature_hash.isin(conflict_hashes).to_numpy()
print("Ambiguous same-feature/different-label rows:", int(conflict_mask.sum()))
if conflict_mask.any():
    keep = ~conflict_mask
    X_raw = X_raw.loc[keep].reset_index(drop=True)
    labels = labels.loc[keep].reset_index(drop=True)
    working = working.loc[keep].reset_index(drop=True)
dup_mask = X_raw.duplicated(keep="first")
print("Exact duplicate feature rows:", int(dup_mask.sum()))
if DROP_EXACT_DUPLICATES:
    keep = ~dup_mask
    X_raw = X_raw.loc[keep].reset_index(drop=True)
    labels = labels.loc[keep].reset_index(drop=True)
    working = working.loc[keep].reset_index(drop=True)
print("Rows after audit:", len(X_raw), "| classes:", labels.value_counts().to_dict())
assert set(labels.unique()) == set(class_order), "One or more classes disappeared during cleanup."
assert not X_raw.duplicated().any() if DROP_EXACT_DUPLICATES else True

### 08 — Split into training and held-out testing records

**Purpose:** Make the **only real-data train/test split**, before fitting filters or scalers. When filenames repeat, split by filename group to avoid grouping leakage; otherwise use stratification. **Expected output:** split method and train/test row counts. Group splitting tries multiple seeds to retain all classes.

In [ ]:
all_idx = np.arange(len(X_raw))
use_groups = ("Filename" in working.columns
              and working["Filename"].notna().any()
              and working["Filename"].duplicated().any())
if use_groups:
    groups = working["Filename"].astype("string").fillna(
        pd.Series(["__missing_row_" + str(i) for i in range(len(working))], dtype="string"))
    best_split, best_drift = None, float("inf")
    for attempt in range(30):
        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=SEED + attempt)
        a, b = next(splitter.split(X_raw, labels, groups=groups))
        if set(labels.iloc[a]) != set(class_order) or set(labels.iloc[b]) != set(class_order):
            continue
        p_full = labels.value_counts(normalize=True).reindex(class_order, fill_value=0)
        p_test = labels.iloc[b].value_counts(normalize=True).reindex(class_order, fill_value=0)
        drift = float((p_full - p_test).abs().sum())
        if drift < best_drift:
            best_split, best_drift = (a, b), drift
    if best_split is None:
        raise RuntimeError("Filename-group split cannot retain all classes. Inspect Filename groups.")
    tr_idx, te_idx = best_split
    split_method = "Filename-group holdout (best of 30 seeds)"
    assert not set(groups.iloc[tr_idx]).intersection(set(groups.iloc[te_idx]))
else:
    tr_idx, te_idx = train_test_split(all_idx, test_size=TEST_FRACTION,
                                    random_state=SEED, stratify=labels)
    split_method = "Stratified holdout"
X_train_raw, X_test_raw = X_raw.iloc[tr_idx].copy(), X_raw.iloc[te_idx].copy()
y_train_text = labels.iloc[tr_idx].to_numpy()
y_test_text = labels.iloc[te_idx].to_numpy()
assert set(class_order) == set(y_train_text) == set(y_test_text)
assert not set(tr_idx).intersection(set(te_idx))
print("Split method:", split_method)
print("Training rows:", len(tr_idx), "| held-out rows:", len(te_idx))

### 09 — Save split indices and encode labels

**Purpose:** Freeze the exact row indices and convert family names to integer labels for XGBoost and SVM. **Expected output:** encoded classes and separate train/test distributions. These indices are reused by every compared model.

In [ ]:
np.savez(OUTPUT_DIR / "frozen_split_indices.npz", train=tr_idx, test=te_idx)
le = LabelEncoder().fit(class_order)
y_train = le.transform(y_train_text)
y_test = le.transform(y_test_text)
print("Split:", split_method, "| train:", len(tr_idx), "| test:", len(te_idx))
print("Train distribution:", pd.Series(y_train_text).value_counts().to_dict())
print("Test distribution:", pd.Series(y_test_text).value_counts().to_dict())

### 10 — Fill missing values using training statistics

**Purpose:** Learn numeric medians on training records only, then use those same medians for the test set and future dashboard inputs. **Expected output:** no missing numeric values after imputation.

In [ ]:
fill_values = X_train_raw.median(numeric_only=True).fillna(0)
Xtr = X_train_raw.fillna(fill_values).copy()
Xte = X_test_raw.fillna(fill_values).copy()

assert not Xtr.isna().any().any() and not Xte.isna().any().any()
print("Missing values after train-only imputation: 0")

### 11 — Remove near-constant features

**Purpose:** Reuse the corrected v3 variance threshold `1e-8`. A feature with almost no variation contributes little information. **Expected output:** count of features retained. This filtering is learned from training data only.

In [ ]:
variances = Xtr.var()
keep_variance = variances[variances > 1e-8].index.tolist()
Xtr = Xtr[keep_variance]
Xte = Xte[keep_variance]

print("Variance filter:", len(raw_feature_cols), "→", len(keep_variance))

### 12 — Remove highly correlated features

**Purpose:** Reuse the v3 absolute correlation threshold `0.95`. Retain a representative feature when two training features are nearly redundant. **Expected output:** final kept column count and dropped column names.

In [ ]:
corr = Xtr.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]
kept_columns = [c for c in keep_variance if c not in drop_corr]
Xtr = Xtr[kept_columns]
Xte = Xte[kept_columns]
assert len(kept_columns) >= N_QUBITS, "Fewer than 8 features remain."

print("Correlation filter:", len(keep_variance), "→", len(kept_columns))
print("Dropped correlated columns:", drop_corr[:15])

### 13 — Apply signed-log transformation and standardization

**Purpose:** Compress extreme numerical ranges with signed-log, then fit `StandardScaler` on training records. **Expected output:** finite train/test arrays. These exact fitted parameters will be saved for inference.

In [ ]:
def signed_log(frame):
    return np.sign(frame) * np.log1p(np.abs(frame))
standard_scaler = StandardScaler().fit(signed_log(Xtr))
Xtr_s = standard_scaler.transform(signed_log(Xtr))
Xte_s = standard_scaler.transform(signed_log(Xte))
assert np.isfinite(Xtr_s).all() and np.isfinite(Xte_s).all()

print("Scaled shapes:", Xtr_s.shape, Xte_s.shape)

### 14 — Train XGBoost feature selector

**Purpose:** Follow the v3 **XGBoost top-eight feature** variant. XGBoost is used to rank the filtered, standardized training features; it is not the final classifier. **Expected output:** the names of eight selected features.

In [ ]:
selector = XGBClassifier(n_estimators=100, max_depth=4, random_state=SEED,
                         eval_metric="mlogloss", tree_method="hist", n_jobs=4)
selector.fit(Xtr_s, y_train)
selected_idx = np.argsort(selector.feature_importances_)[::-1][:N_QUBITS]
selected_feature_names = [kept_columns[int(i)] for i in selected_idx]

assert len(selected_idx) == N_QUBITS
print("Selected features:", selected_feature_names)

### 15 — Map selected features to quantum rotation angles

**Purpose:** Fit `RobustScaler` on the selected training features, clip to `[-3,3]`, and map to `[-π,π]` exactly as corrected v3.1 does. **Expected output:** finite eight-dimensional angles for every row.

In [ ]:
robust_scaler = RobustScaler().fit(Xtr_s[:, selected_idx])
def to_angles(z):
    return np.clip(robust_scaler.transform(z[:, selected_idx]), -3, 3) / 3 * np.pi
Xtr_q_full = to_angles(Xtr_s)
Xte_q_full = to_angles(Xte_s)
print("Top features:", selected_feature_names)
print("Angle range:", Xtr_q_full.min(), Xtr_q_full.max())
assert Xtr_q_full.shape[1] == N_QUBITS and Xte_q_full.shape[1] == N_QUBITS
assert np.isfinite(Xtr_q_full).all() and np.isfinite(Xte_q_full).all()

### 16 — Define a reproducible, stratified quantum subset helper

**Purpose:** Quantum kernel simulation is expensive, so we use a controlled sample budget. This helper preserves class representation when selecting a smaller subset. **Expected output:** a function definition; no training occurs yet.

In [ ]:
def stratified_subset_indices(y, size, seed):
    if size is None or size >= len(y):
        return np.arange(len(y))
    if size < len(np.unique(y)) * 2:
        raise ValueError("Sample budget too small for stratification.")
    idx, _ = train_test_split(np.arange(len(y)), train_size=size,
                              stratify=y, random_state=seed)
    return np.sort(idx)

### 17 — Select matched quantum and classical samples

**Purpose:** Choose fixed training and test row indices **once**. QSVM, RBF-SVM and Random Forest will use the same rows and labels. **Expected output:** subset sizes and class counts, with indices saved for reproducibility.

In [ ]:
train_sel = stratified_subset_indices(y_train, TRAIN_SUBSET, SEED + 101)
test_sel = stratified_subset_indices(y_test, TEST_SUBSET, SEED + 202)
Xq_train, Xq_test = Xtr_q_full[train_sel], Xte_q_full[test_sel]
yq_train, yq_test = y_train[train_sel], y_test[test_sel]
np.savez(OUTPUT_DIR / "quantum_subset_indices.npz", train=train_sel, test=test_sel)
print("Matched experiment:", len(train_sel), "train,", len(test_sel), "test")
print("Test counts:", dict(zip(*np.unique(le.inverse_transform(yq_test), return_counts=True))))

### 18 — Choose PennyLane simulator and measurement pairs

**Purpose:** Use `lightning.qubit` for faster CPU state-vector simulation if available, otherwise `default.qubit`. Create four non-overlapping qubit pairs for eight qubits. **Expected output:** simulator name and pair indices.

In [ ]:
try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    DEVICE_NAME = "lightning.qubit"
except Exception as exc:
    warnings.warn(f"Lightning unavailable ({exc}); using slower default.qubit.")
    dev = qml.device("default.qubit", wires=N_QUBITS)
    DEVICE_NAME = "default.qubit"

pairs = [(i, i + 1) for i in range(0, N_QUBITS - 1, 2)]
state_ids = np.arange(2 ** N_QUBITS)
pair_masks = [
    (((state_ids >> (N_QUBITS - 1 - a)) & 1) == 0) &
    (((state_ids >> (N_QUBITS - 1 - b)) & 1) == 0)
    for a, b in pairs
]


print("Device:", DEVICE_NAME, "| pairs:", pairs)

### 19 — Build corrected v3.1 quantum overlap circuit

**Purpose:** Implement the **bug-fixed** data-reuploading circuit. The trainable `RY` rotations occur inside encoding layers so they do not cancel immediately. **Expected output:** circuit and local similarity function definitions.

In [ ]:
@qml.qnode(dev)
def overlap_probs(x1, x2, w):
    for _ in range(LAYERS):
        qml.AngleEmbedding(x1, wires=range(N_QUBITS), rotation="Y")
        for i in range(N_QUBITS):
            qml.RY(w[i], wires=i)
        for i in range(N_QUBITS):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS])
    for _ in range(LAYERS):
        for i in reversed(range(N_QUBITS)):
            qml.CNOT(wires=[i, (i + 1) % N_QUBITS])
        for i in range(N_QUBITS):
            qml.RY(-w[i], wires=i)
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(N_QUBITS), rotation="Y")
    return qml.probs(wires=range(N_QUBITS))

def local_similarity(x1, x2, w):
    probs = np.asarray(overlap_probs(x1, x2, w))
    return float(np.mean([probs[mask].sum() for mask in pair_masks]))

### 20 — Test basic quantum similarity behavior

**Purpose:** Check the circuit on one identical input and two different inputs. **Expected output:** self-similarity close to 1, two directional similarity values, and a sensitivity diagnostic for trainable weights. **Note:** the local readout is not guaranteed symmetric or PSD.

In [ ]:
w_probe = 0.05 * np.random.default_rng(SEED).standard_normal(N_QUBITS)
diag = local_similarity(Xq_train[0], Xq_train[0], w_probe)
print("Self similarity:", diag)
assert np.isclose(diag, 1.0, atol=1e-6), "Overlap circuit failed K(x,x)=1 check."
k12 = local_similarity(Xq_train[0], Xq_train[1], w_probe)
k21 = local_similarity(Xq_train[1], Xq_train[0], w_probe)
print("Symmetry diagnostic:", k12, k21, "| abs difference:", abs(k12-k21))
print("Trainable-weight sensitivity:",
      abs(k12 - local_similarity(Xq_train[0], Xq_train[1], w_probe + 0.4)))
print("Note: local overlap similarity is not automatically a Mercer/PSD kernel.")

### 21 — Quantum preflight before expensive loops

**Purpose:** Run numerical assertions for input shapes, angle ranges, labels, probabilities and a small 3×3 similarity matrix. **Expected output:** `Quantum preflight PASSED`. If this fails, **stop** and fix it before training.

In [ ]:
# Small quantum preflight before expensive KTA/Nyström loops
assert Xq_train.shape == (len(yq_train), N_QUBITS)
assert Xq_test.shape == (len(yq_test), N_QUBITS)
assert np.isfinite(Xq_train).all() and np.isfinite(Xq_test).all()
assert np.max(np.abs(Xq_train)) <= np.pi + 1e-12
assert np.max(np.abs(Xq_test)) <= np.pi + 1e-12
assert len(np.unique(yq_train)) == len(le.classes_)
assert len(np.unique(yq_test)) == len(le.classes_)

start = time.perf_counter()
preflight_K = np.array([
    [local_similarity(a, b, w_probe) for b in Xq_train[:3]]
    for a in Xq_train[:3]
])
elapsed = time.perf_counter() - start
assert np.isfinite(preflight_K).all()
assert ((preflight_K >= -1e-8) & (preflight_K <= 1 + 1e-8)).all()
assert np.allclose(np.diag(preflight_K), 1.0, atol=1e-6)
print("Quantum preflight PASSED.")
print("3×3 local similarity matrix:\n", np.round(preflight_K, 5))
print("Observed asymmetry (not necessarily a bug for this local readout):",
      float(np.max(np.abs(preflight_K - preflight_K.T))))
print("Preflight runtime:", round(elapsed, 3), "seconds")
print("Approximate raw kernel calls for training:",
      len(Xq_train) * N_LANDMARKS + len(Xq_test) * N_LANDMARKS + N_LANDMARKS**2)
print("This preflight checks the circuit and numerical outputs, not final model accuracy.")

### 22 — Select a training-only KTA subset

**Purpose:** Choose the examples used to tune quantum rotations. Their labels come exclusively from the training subset, never from held-out data. **Expected output:** alignment subset shape and class distribution.

In [ ]:
align_rng = np.random.default_rng(SEED + 10_000)
align_idx = align_rng.choice(len(Xq_train), min(ALIGN_SUBSET, len(Xq_train)), replace=False)
X_align, y_align = Xq_train[align_idx], yq_train[align_idx]
Y_target = np.where(y_align[:, None] == y_align[None, :], 1.0, -1.0)

print("KTA subset:", X_align.shape, "| labels:", dict(zip(*np.unique(y_align, return_counts=True))))

### 23 — Define kernel-target-alignment objective

**Purpose:** Construct the alignment objective measuring agreement between local similarity values and label similarity. **Expected output:** a callable optimization loss; this cell does not optimize yet.

In [ ]:
calls = [0]
def kta_loss(w):
    calls[0] += 1
    K = np.array([[local_similarity(a, b, w) for b in X_align] for a in X_align])
    denom = np.linalg.norm(K) * np.linalg.norm(Y_target) + 1e-9
    return -float(np.sum(K * Y_target) / denom)

### 24 — Optimize quantum rotation weights

**Purpose:** Use the corrected v3.1 **COBYLA** procedure to optimize trainable angles. **Expected output:** learned weights, alignment objective, number of evaluations and elapsed time. Optimization may be slow.

In [ ]:
w0 = 0.05 * np.random.default_rng(SEED).standard_normal(N_QUBITS)
t0 = time.perf_counter()
opt = minimize(kta_loss, w0, method="COBYLA",
               options={"maxiter": max(ALIGN_MAXITER, N_QUBITS + 2), "rhobeg": 0.3})
w_trained = np.asarray(opt.x, dtype=float)
align_seconds = time.perf_counter() - t0
print("KTA:", {"objective": -float(opt.fun), "calls": calls[0],
               "seconds": round(align_seconds, 1), "success": bool(opt.success)})
print("Trained weights:", np.round(w_trained, 4))

### 25 — Select Nyström landmarks and similarity helper

**Purpose:** Choose fixed training-only landmark examples and define a function to compute similarities. **Expected output:** landmark count. Nyström reduces the number of quantum circuit calls compared with a full all-pairs kernel.

In [ ]:
land_rng = np.random.default_rng(SEED + 20_000)
landmark_idx = land_rng.choice(len(Xq_train), min(N_LANDMARKS, len(Xq_train)), replace=False)
landmarks = Xq_train[landmark_idx].copy()

def cross_similarity(A, B, weights, log_every=0):
    out = np.empty((len(A), len(B)), dtype=float)
    for i, a in enumerate(A):
        for j, b in enumerate(B):
            out[i, j] = local_similarity(a, b, weights)
        if log_every and (i + 1) % log_every == 0:
            print(f"  {i+1}/{len(A)} rows")
    return out


print("Landmarks selected:", len(landmarks))

### 26 — Compute quantum similarities to landmarks

**Purpose:** Evaluate train-to-landmark, test-to-landmark, and landmark-to-landmark similarities using the **trained** rotations. **Expected output:** three matrices and progress messages. This is usually the slowest cell.

In [ ]:
t0 = time.perf_counter()
K_train_land = cross_similarity(Xq_train, landmarks, w_trained, log_every=100)
K_test_land = cross_similarity(Xq_test, landmarks, w_trained, log_every=60)
K_mm_raw = cross_similarity(landmarks, landmarks, w_trained)

print("Quantum matrix shapes:", K_train_land.shape, K_test_land.shape, K_mm_raw.shape)

### 27 — Stabilize Nyström features and inspect eigenvalues

**Purpose:** Symmetrize the landmark block, report its eigenvalues, and regularize the inverse square root. **Expected output:** a finite Nyström transformation and training/test features. **Important:** this numerical fix does not prove the local readout is a valid PSD kernel.

In [ ]:
K_mm = (K_mm_raw + K_mm_raw.T) / 2.0
eigvals, eigvecs = np.linalg.eigh(K_mm)
print("Landmark matrix: max asymmetry =", float(np.max(np.abs(K_mm_raw-K_mm_raw.T))))
print("Eigenvalue range:", float(eigvals.min()), float(eigvals.max()),
      "| negative eigenvalues:", int((eigvals < -1e-8).sum()))
eigvals_clipped = np.clip(eigvals, 1e-8, None)
K_mm_inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals_clipped)) @ eigvecs.T
psi_train = K_train_land @ K_mm_inv_sqrt
psi_test = K_test_land @ K_mm_inv_sqrt

assert np.isfinite(psi_train).all() and np.isfinite(psi_test).all()
print("Nyström feature dimensions:", psi_train.shape, psi_test.shape)

### 28 — Train the final linear SVM on quantum features

**Purpose:** Complete the hybrid architecture by fitting a classical linear SVM on Nyström features derived from the quantum circuit. **Expected output:** the trained QSVM, test predictions and kernel-stage runtime.

In [ ]:
qsvm = SVC(kernel="linear", C=SVM_C)
qsvm.fit(psi_train, yq_train)
pred_qsvm = qsvm.predict(psi_test)
kernel_seconds = time.perf_counter() - t0
print("QSVM finished in", round(kernel_seconds, 1), "seconds",
      "| feature shape:", psi_train.shape)

### 29 — Train matched classical baselines

**Purpose:** Train RBF-SVM and Random Forest on the **same eight selected features and exact same rows**. Train a second RF on all filtered features to show the cost of reducing to eight features. **Expected output:** three trained classical models.

In [ ]:
svm_rbf = SVC(kernel="rbf", C=SVM_C, gamma="scale")
rf_8 = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_all = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
svm_rbf.fit(Xq_train, yq_train)
rf_8.fit(Xq_train, yq_train)
rf_all.fit(Xtr_s[train_sel], yq_train)

### 30 — Predict with classical baselines

**Purpose:** Evaluate the classical models on the identical held-out test subset used by the QSVM. **Expected output:** three prediction arrays.

In [ ]:
pred_rbf = svm_rbf.predict(Xq_test)
pred_rf8 = rf_8.predict(Xq_test)
pred_rfall = rf_all.predict(Xte_s[test_sel])

### 31 — Compare all models fairly

**Purpose:** Report accuracy, Macro-F1 and balanced accuracy for each model using identical labels and test rows. **Expected output:** comparison table and CSV file. No result is claimed until this cell runs.

In [ ]:
predictions = {
    "QSVM v3.1 local": pred_qsvm,
    "RBF-SVM same 8": pred_rbf,
    "RF same 8": pred_rf8,
    "RF all filtered": pred_rfall,
}
rows = []
for name, pred in predictions.items():
    rows.append({"model": name, "accuracy": accuracy_score(yq_test, pred),
                 "macro_f1": f1_score(yq_test, pred, average="macro", zero_division=0),
                 "balanced_accuracy": balanced_accuracy_score(yq_test, pred),
                 "n_train": len(yq_train), "n_test": len(yq_test)})
metrics_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
display(metrics_df)
metrics_df.to_csv(OUTPUT_DIR / "matched_model_metrics.csv", index=False)
print("NOTE: these are NEW real-data results; do not substitute historical CTGAN scores.")

### 32 — Print detailed QSVM classification report

**Purpose:** Inspect precision, recall and F1 for each malware family rather than relying on one headline accuracy. **Expected output:** a per-class classification report.

In [ ]:
print("QSVM classification report:")
print(classification_report(yq_test, pred_qsvm, target_names=list(le.classes_), zero_division=0))

### 33 — Save confusion matrix and comparison chart

**Purpose:** Produce visual evidence of QSVM errors and the classical-versus-quantum Macro-F1 comparison. **Expected output:** `evaluation.png` in the output folder.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(yq_test, pred_qsvm,
    display_labels=le.classes_, ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Corrected QSVM v3.1 — held-out real data")
axes[1].barh(metrics_df["model"], metrics_df["macro_f1"], color="#4466a8")
axes[1].set_xlim(0, 1)
axes[1].set_xlabel("Macro-F1")
axes[1].set_title("Matched subset comparison")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "evaluation.png", dpi=180, bbox_inches="tight")
plt.show()

### 34 — Export evaluation metadata

**Purpose:** Save task, seed, dataset fingerprint, selected features, training budget, quantum runtime and metrics in JSON. **Expected output:** `evaluation_metrics.json`.

In [ ]:
evaluation = {
    "task": TASK, "seed": SEED, "dataset_sha256": sha,
    "n_train_subset": len(yq_train), "n_test_subset": len(yq_test),
    "n_train_pool": len(y_train), "n_test_pool": len(y_test),
    "classes": list(le.classes_), "selected_features": selected_feature_names,
    "device": DEVICE_NAME, "quantum_seconds": kernel_seconds,
    "kta_seconds": align_seconds,
    "metrics": rows,
    "warnings": ["No test-set tuning", "Local similarity may be indefinite",
                 "Quantum test subset is smaller than full real-data holdout"]
}
(OUTPUT_DIR / "evaluation_metrics.json").write_text(json.dumps(evaluation, indent=2))
print("Saved evaluation files to", OUTPUT_DIR)

### 35 — Assemble the complete saved-model bundle

**Purpose:** Collect all fitted preprocessing objects, quantum circuit configuration, trained KTA weights, Nyström landmarks and SVM into one dictionary. **Expected output:** a bundle ready for serialization. The PennyLane QNode is reconstructed during inference.

In [ ]:
bundle = {
    "format_version": "qtagger-v3.1-1",
    "task": TASK,
    "dataset_sha256": sha,
    "classes": list(le.classes_),
    "raw_feature_columns": raw_feature_cols,
    "fill_values": fill_values.to_dict(),
    "kept_columns": kept_columns,
    "removed_correlated_columns": drop_corr,
    "standard_scaler": standard_scaler,
    "selected_indices": selected_idx.astype(int),
    "selected_feature_names": selected_feature_names,
    "robust_scaler": robust_scaler,
    "quantum_config": {
        "n_qubits": N_QUBITS, "layers": LAYERS,
        "pairs": pairs, "device": DEVICE_NAME, "seed": SEED
    },
    "kta_weights": w_trained,
    "landmarks": landmarks,
    "K_mm_inv_sqrt": K_mm_inv_sqrt,
    "svm": qsvm,
    "evaluation": evaluation,
}

### 36 — Save model and environment versions

**Purpose:** Write the full model to `.joblib` and save package versions. **Expected output:** `qtagger_v3_1.joblib` and `environment.json`. Only load joblib files from trusted sources.

In [ ]:
model_path = OUTPUT_DIR / "qtagger_v3_1.joblib"
joblib.dump(bundle, model_path, compress=3)
environment = {"python": platform.python_version(), "numpy": np.__version__,
               "pandas": pd.__version__, "scipy": scipy.__version__,
               "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
               "pennylane": qml.__version__, "joblib": joblib.__version__}
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
print("Saved:", model_path, "| size:", round(model_path.stat().st_size/1024, 1), "KiB")

### 37 — Define raw-feature inference function

**Purpose:** Reconstruct the corrected circuit from the saved configuration, then apply **the exact same** imputation, filtering, scaling, feature selection, quantum mapping and final SVM. **Expected output:** a `make_predictor` function for dashboard integration.

In [ ]:
def make_predictor(saved):
    qcfg = saved["quantum_config"]
    nq, depth = int(qcfg["n_qubits"]), int(qcfg["layers"])
    try:
        predict_dev = qml.device("lightning.qubit", wires=nq)
    except Exception:
        predict_dev = qml.device("default.qubit", wires=nq)
    masks = []
    ids = np.arange(2**nq)
    for a, b in qcfg["pairs"]:
        masks.append((((ids >> (nq-1-a)) & 1) == 0) &
                     (((ids >> (nq-1-b)) & 1) == 0))
    @qml.qnode(predict_dev)
    def probs(x1, x2, w):
        for _ in range(depth):
            qml.AngleEmbedding(x1, wires=range(nq), rotation="Y")
            for i in range(nq):
                qml.RY(w[i], wires=i)
            for i in range(nq):
                qml.CNOT(wires=[i, (i+1) % nq])
        for _ in range(depth):
            for i in reversed(range(nq)):
                qml.CNOT(wires=[i, (i+1) % nq])
            for i in range(nq):
                qml.RY(-w[i], wires=i)
            qml.adjoint(qml.AngleEmbedding)(x2, wires=range(nq), rotation="Y")
        return qml.probs(wires=range(nq))
    def predict(raw_df):
        missing = sorted(set(saved["raw_feature_columns"]) - set(raw_df.columns))
        if missing:
            raise ValueError(f"Missing {len(missing)} raw feature columns: {missing[:12]}")
        X = raw_df[saved["raw_feature_columns"]].copy()
        X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        X = X.fillna(pd.Series(saved["fill_values"]))
        X = X[saved["kept_columns"]]
        X = np.sign(X) * np.log1p(np.abs(X))
        z = saved["standard_scaler"].transform(X)
        z = z[:, saved["selected_indices"]]
        z = np.clip(saved["robust_scaler"].transform(z), -3, 3) / 3 * np.pi
        weights = saved["kta_weights"]
        landmarks = saved["landmarks"]
        K = np.array([[float(np.mean([np.asarray(probs(x, lp, weights))[m].sum()
                                       for m in masks])) for lp in landmarks] for x in z])
        psi = K @ saved["K_mm_inv_sqrt"]
        pred_idx = saved["svm"].predict(psi)
        return np.array(saved["classes"], dtype=object)[pred_idx]
    return predict

### 38 — Reload model and verify predictions

**Purpose:** Load the model from disk and predict on five held-out real records. Compare against predictions from the in-memory model. **Expected output:** `Reload test passed`; any mismatch raises an error and must be fixed before building the dashboard.

In [ ]:
loaded = joblib.load(model_path)
predict_reloaded = make_predictor(loaded)
smoke_raw = X_test_raw.iloc[test_sel[:5]].copy()
pred_from_disk = predict_reloaded(smoke_raw)
expected = le.inverse_transform(pred_qsvm[:5])
assert np.array_equal(pred_from_disk, expected), (
    f"Reload mismatch: {pred_from_disk} vs {expected}"
)
print("Reload test passed — predictions match original QSVM.")
print(pd.DataFrame({"true": y_test_text[test_sel[:5]],
                    "predicted": pred_from_disk}))

### 39 — Export a demo CSV and predicted labels

**Purpose:** Save five held-out numerical feature records without their labels, plus the model's predictions. **Expected output:** `demo_unlabeled_features.csv` and `demo_predictions.csv` for dashboard testing.

In [ ]:
demo_csv = OUTPUT_DIR / "demo_unlabeled_features.csv"
smoke_raw.to_csv(demo_csv, index=False)
pd.DataFrame({"predicted_family": pred_from_disk}).to_csv(
    OUTPUT_DIR / "demo_predictions.csv", index=False)
print("Demo CSV:", demo_csv)
print("Predictions:", OUTPUT_DIR / "demo_predictions.csv")
print("Final model:", model_path)

### 40 — Package Kaggle outputs into one downloadable ZIP

**Purpose:** Bundle the trained model, environment versions, evaluation reports, figures and demo CSV. **Expected output:** `qtagger_plus_kaggle_artifacts.zip` under `/kaggle/working` on Kaggle; download it from the notebook's **Output** tab.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive_path = OUTPUT_DIR.parent / "qtagger_plus_kaggle_artifacts.zip"
with ZipFile(archive_path, "w", ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUTPUT_DIR.parent)))
print("Downloadable artifact ZIP:", archive_path)
print("ZIP size (MiB):", round(archive_path.stat().st_size / (1024**2), 2))
print("Notebook complete. Next stage: Streamlit dashboard using the saved model.")

## What you should have after **all** cells succeed

| Artifact | Why it exists |
|---|---|
| `qtagger_v3_1.joblib` | Trained hybrid QSVM, preprocessing, quantum weights and Nyström transform |
| `environment.json` | Package versions needed for compatible reloading |
| `evaluation_metrics.json` | Task configuration, class labels, metrics and runtime |
| `matched_model_metrics.csv` | Fair comparison against classical SVM and RF |
| `evaluation.png` | Confusion matrix and Macro-F1 chart |
| `frozen_split_indices.npz` | Real-data training/test split indices |
| `quantum_subset_indices.npz` | Exact QSVM/classical comparison indices |
| `demo_unlabeled_features.csv` | Unlabeled raw memory-feature rows for a dashboard |
| `demo_predictions.csv` | Inference results from the reloaded model |
| `qtagger_plus_kaggle_artifacts.zip` | One-click collection of all artifacts |

**Important limits:** No accuracy can be guaranteed before execution. This notebook does not create a Streamlit dashboard, scan executable files, or retrain synthetic generators. A full scientific evaluation should also use multiple seeds and more held-out test records. If you change the task, preprocessing, seed or architecture, retrain and resave the bundle.